In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Settings
NUM_ROWS = 100000
DEVICES = [f"can_{str(i).zfill(3)}" for i in range(1, 21)]
START_DATE = datetime(2026, 1, 19) # A complete week in Jan 2026

# Sudan Community Midpoints (Lat, Lon)
COMMUNITIES = [
    (15.5007, 32.5599), (15.6133, 32.5322), (14.4015, 33.5198), 
    (13.1747, 30.2097), (12.8628, 32.9838), (14.0000, 31.0000),
    (15.0000, 35.0000), (13.5000, 34.0000), (12.5000, 30.5000),
    (15.8000, 33.2000), (14.2000, 32.1000), (13.9000, 35.5000),
    (14.8000, 34.5000), (15.2000, 31.8000), (12.9000, 33.9000),
    (13.2000, 31.2000), (14.5000, 33.1000), (15.4000, 32.8000),
    (13.7000, 30.8000), (15.1000, 33.6000)
]

data = []

for i in range(NUM_ROWS):
    dev = random.choice(DEVICES)
    msg = random.choices(["EVENT", "HEARTBEAT"], weights=[0.4, 0.6])[0]
    
    # Event logic
    event = ""
    if msg == "EVENT":
        event = random.choice(["REFILL", "SERVICE_STOP", "DEPLETED"])
        fix, hdop, sats = "3D", round(random.uniform(0.8, 1.5), 1), random.randint(9, 14)
    else:
        fix = random.choice(["2D", "3D", "NO_FIX"])
        hdop = round(random.uniform(2.0, 8.0), 1) if fix != "3D" else 1.8
        sats = random.randint(3, 7) if fix != "3D" else 9
    
    # Time logic (Spread across 1 week)
    seconds_offset = random.randint(0, 604800)
    ts = (START_DATE + timedelta(seconds=seconds_offset)).isoformat() + "Z"
    
    # Pressure logic (Pa) - Higher is more water (compressing air)
    if event == "REFILL":
        pres = random.randint(109000, 112000)
    elif event == "DEPLETED":
        pres = 101325
    else:
        pres = random.randint(101500, 108000)

    # Location Logic
    if random.random() > 0.1: # 90% in communities
        base_lat, base_lon = random.choice(COMMUNITIES)
        lat = base_lat + random.uniform(-0.009, 0.009) # ~1km radius
        lon = base_lon + random.uniform(-0.009, 0.009)
    else: # 10% random in Sudan range
        lat = random.uniform(12.3, 15.7)
        lon = random.uniform(30.2, 35.8)

    data.append([
        dev, ts, pres, round(random.uniform(3.5, 4.2), 2),
        round(lat, 5), round(lon, 5), hdop, sats, fix,
        round(random.uniform(0, 1.5), 2) if event != "" else round(random.uniform(0, 12), 2),
        msg, event
    ])

cols = ["device_id", "recorded_at", "pressure_pa", "battery_v", "lat", "lon", "hdop", "sat_count", "fix_type", "speed_mps", "msg_type", "event_type"]
df = pd.DataFrame(data, columns=cols).sort_values(by=["device_id", "recorded_at"])

df.to_csv("/Users/xDAyN/Desktop/Lifelines Hackathon/newly_generated_data.csv", index=False)
print("File 'sudan_water_tracking.csv' created with 10000 rows.")

File 'sudan_water_tracking.csv' created with 10000 rows.
